# Home 
Receta comun de los experimentos colaborativos (comision lunes, DMEyF 2026).

Todo lo que NO cambia entre experimentos vive aca: carga de datos, lags y
deltas, la optimizacion bayesiana, la ganancia y los graficos.

Cada experimento es una notebook que importa este modulo y cambia UNA cosa
(por ejemplo, deflactar antes de armar las historicas).

    import comun
    df = comun.cargar_datos()
    datos, features = comun.agregar_lags_deltas(df)
    estudio = comun.optimizar(datos, features, "baseline")
    res, curvas, probas = comun.correr_semillas(datos, features, estudio)


# Set up

In [1]:
import time
import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd

In [2]:
PARQUET = "/home/ds/datasets/datos_2019.parquet"
STORAGE = "sqlite:////home/ds/datasets/optuna.db"

SEMILLAS = [100129, 100183, 100151, 500041, 900007]

GANANCIA_ACIERTO = 1_072_500
COSTO_ESTIMULO = 27_500

MESES_TRAIN = [201903, 201904, 201905, 201906]
MES_VALID = 201908
MESES_TRAIN_FINAL = [201903, 201904, 201905, 201906, 201907, 201908]
MES_TEST = 201910

NO_FEATURES = ("numero_de_cliente", "foto_mes", "clase_ternaria")

# Datos

In [3]:
def cargar_datos(parquet=PARQUET):
    """El dataset 2019 con clase_ternaria, ordenado por cliente y mes."""
    df = pd.read_parquet(parquet)
    return df.sort_values(["numero_de_cliente", "foto_mes"]).reset_index(drop=True)


def columnas_base(df):
    return [c for c in df.columns if c not in NO_FEATURES]


def agregar_lags_deltas(df):
    """lag y delta 1 y 2 por cliente. Devuelve (datos, features).

    Ojo con el orden: si el experimento deflacta, deflacta ANTES de llamar
    a esta funcion, para que las historicas se calculen sobre valores reales.
    """
    base = columnas_base(df)
    valores = df[base].astype("float32")
    g = valores.groupby(df["numero_de_cliente"], sort=False)

    piezas = [df[list(NO_FEATURES)], valores]
    for k in (1, 2):
        lag = g.shift(k)
        piezas.append(lag.add_suffix(f"_lag{k}"))
        piezas.append((valores - lag).add_suffix(f"_delta{k}"))
        del lag

    datos = pd.concat(piezas, axis=1)
    features = [c for c in datos.columns if c not in NO_FEATURES]
    return datos, features


def target(clase):
    """B+1 y B+2 combinadas, como pide la consigna del baseline."""
    return clase.isin(["BAJA+1", "BAJA+2"]).astype("int8")

# Optimizacion (BO)

In [4]:
def optimizar(datos, features, nombre, n_trials=50, semilla=SEMILLAS[0],
              storage=STORAGE, meses_train=None, mes_valid=MES_VALID):
    """BO sobre los 5 hiperparametros de la consigna.

    Idempotente: si el estudio ya tiene n_trials completos, no corre nada,
    asi la notebook soporta Run All sin repetir horas de computo.
    """
    meses_train = meses_train or MESES_TRAIN

    tr = datos[datos.foto_mes.isin(meses_train)]
    va = datos[datos.foto_mes == mes_valid]

    dtrain = lgb.Dataset(tr[features], label=target(tr.clase_ternaria),
                         free_raw_data=False, params={"feature_pre_filter": False})
    dvalid = lgb.Dataset(va[features], label=target(va.clase_ternaria), reference=dtrain,
                         free_raw_data=False, params={"feature_pre_filter": False})

    def objetivo(trial):
        params = {
            "objective": "binary",
            "metric": "auc",
            "verbosity": -1,
            "seed": semilla,
            "bagging_freq": 1,
            "feature_pre_filter": False,
            "num_leaves": trial.suggest_int("num_leaves", 8, 256),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 100, 8000, log=True),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.2, 1.0),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.2, 1.0),
        }
        modelo = lgb.train(params, dtrain, num_boost_round=2000, valid_sets=[dvalid],
                           callbacks=[lgb.early_stopping(100, verbose=False)])
        trial.set_user_attr("mejor_iteracion", modelo.best_iteration)
        return modelo.best_score["valid_0"]["auc"]

    estudio = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=semilla),
        study_name=nombre,
        storage=storage,
        load_if_exists=True,
    )

    completos = sum(1 for t in estudio.trials if t.state.name == "COMPLETE")
    faltan = max(0, n_trials - completos)
    print(f"[{nombre}] trials completos: {completos}, faltan: {faltan}")
    if faltan:
        estudio.optimize(objetivo, n_trials=faltan, show_progress_bar=True)

    print(f"[{nombre}] mejor AUC {estudio.best_value:.5f}")
    print(f"[{nombre}] {estudio.best_params}")
    return estudio


# Ganancia

In [5]:
def curva_ganancia(clases, prob):
    """Ganancia acumulada ordenando los clientes por probabilidad."""
    orden = np.argsort(-prob)
    return np.cumsum(np.where(clases[orden] == "BAJA+2", GANANCIA_ACIERTO, -COSTO_ESTIMULO))


def correr_semilla(datos, features, estudio, semilla,
                   meses=None, mes_test=MES_TEST):
    """Entrena el modelo final con una semilla y mide ganancia en el test."""
    meses = meses or MESES_TRAIN_FINAL

    params = {"objective": "binary", "metric": "auc", "verbosity": -1,
              "seed": semilla, "bagging_freq": 1, **estudio.best_params}
    rondas = estudio.best_trial.user_attrs["mejor_iteracion"]

    trf = datos[datos.foto_mes.isin(meses)]
    modelo = lgb.train(params,
                       lgb.Dataset(trf[features], label=target(trf.clase_ternaria)),
                       num_boost_round=rondas)

    te = datos[datos.foto_mes == mes_test]
    prob = modelo.predict(te[features])
    acum = curva_ganancia(te.clase_ternaria.to_numpy(), prob)
    return prob, acum, int(np.argmax(acum)) + 1, float(acum.max())


def correr_semillas(datos, features, estudio, semillas=None, **kwargs):
    """Corre el modelo final con varias semillas. Devuelve (res, curvas, probas)."""
    semillas = semillas or SEMILLAS
    filas, curvas, probas = [], {}, {}

    for s in semillas:
        t = time.time()
        prob, acum, envios, plata = correr_semilla(datos, features, estudio, s, **kwargs)
        curvas[s], probas[s] = acum, prob
        filas.append({"semilla": s, "ganancia": plata, "envios": envios})
        print(f"  semilla {s}: ${plata:,.0f} con {envios:,} envios ({time.time()-t:.0f}s)")

    res = pd.DataFrame(filas)
    print(f"media ${res.ganancia.mean():,.0f}  desvio ${res.ganancia.std():,.0f}")
    return res, curvas, probas


# Gráficos

In [6]:
def grafico_ganancia(curvas, titulo="Curva de ganancia en 201910", hasta=20000):
    import plotly.graph_objects as go

    fig = go.Figure()
    for nombre, acum in curvas.items():
        fig.add_trace(go.Scatter(y=acum[:hasta], mode="lines", name=str(nombre)))
    fig.update_layout(title=titulo, template="plotly_white",
                      xaxis_title="Clientes estimulados (ordenados por probabilidad)",
                      yaxis_title="Ganancia acumulada")
    return fig


def grafico_roc(datos, probas, mes_test=MES_TEST, titulo="ROC en 201910 (positivo = BAJA+2)"):
    import plotly.graph_objects as go
    from sklearn.metrics import roc_auc_score, roc_curve

    te = datos[datos.foto_mes == mes_test]
    y = (te.clase_ternaria == "BAJA+2").astype(int)

    fig = go.Figure()
    for nombre, prob in probas.items():
        fpr, tpr, _ = roc_curve(y, prob)
        fig.add_trace(go.Scatter(x=fpr, y=tpr, mode="lines",
                                 name=f"{nombre} (AUC {roc_auc_score(y, prob):.4f})"))
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", name="azar",
                             line=dict(dash="dash", color="gray")))
    fig.update_layout(title=titulo, template="plotly_white",
                      xaxis_title="Falsos positivos", yaxis_title="Verdaderos positivos")
    return fig


def grafico_comparacion(resultados, titulo="Ganancia por experimento"):
    """resultados: dict {nombre: DataFrame de correr_semillas}. Boxplot comparativo."""
    import plotly.graph_objects as go

    fig = go.Figure()
    for nombre, res in resultados.items():
        fig.add_trace(go.Box(y=res.ganancia, name=nombre, boxpoints="all", jitter=0.3))
    fig.update_layout(title=titulo, template="plotly_white", yaxis_title="Ganancia en 201910")
    return fig


# Comparación

In [7]:
def comparar(res_a, res_b, nombre_a="baseline", nombre_b="experimento"):
    """Wilcoxon pareado por semilla, que es como se comparan las distribuciones."""
    from scipy.stats import wilcoxon

    a = res_a.sort_values("semilla").ganancia.to_numpy()
    b = res_b.sort_values("semilla").ganancia.to_numpy()
    dif = b - a

    print(f"{nombre_a}: ${a.mean():,.0f}   {nombre_b}: ${b.mean():,.0f}")
    print(f"diferencia media: ${dif.mean():,.0f}  ({dif.mean()/a.mean()*100:+.2f}%)")
    print(f"gana el experimento en {int((dif > 0).sum())} de {len(dif)} semillas")

    if len(dif) >= 6:
        stat, p = wilcoxon(b, a)
        print(f"Wilcoxon pareado: p = {p:.4f}")
    else:
        print("Wilcoxon necesita al menos 6 pares para dar un p confiable")
    return dif
